# 28 Floyd 算法求解最短路径问题

> **模块**：图论与网络 ｜ **数据**：无（内置示例） ｜ **依赖**：numpy

## 1. 算法描述

Floyd 算法用于寻找图中每个顶点到其他顶点的最短路径（多源最短路径问题），是一种典型的动态规划算法。在阅读前，请先了解图的数据结构「邻接矩阵」。Floyd 算法通过「贪心」和「动态规划」的思想，求解图中所有顶点对之间的最短路径，时间复杂度为 O(n^3)。它能够处理带权重的图，包括负权重的边，但无法处理包含负权重环的图。


## 2. Floyd 算法步骤
1. **初始化距离矩阵**：创建一个二维数组 `dist`，用于表示从每个顶点到其他顶点的初始距离。
2. **更新距离**：通过三个嵌套循环更新距离矩阵，使得 `dist[i][j]` 表示顶点 `i` 到顶点 `j` 的最短路径。
3. **输出结果**：最后输出每对顶点之间的最短路径。

**要点：**
- 以每个点为「中转站」，刷新所有「入度」和「出度」的距离。


## 3. 原理

- 一个点 `i` 到另一个点 `j` 的最短路径有两种可能：
1. 直接从 `i` 到 `j` $(i \rightarrow j)$。
2. 通过一个或多个其他点（k1, k2, …, kn）到达 $(i \rightarrow k1 \rightarrow k2 \rightarrow \ldots \rightarrow j)$。

- 找最短路径时，直接比较这两种方式，选择更短的一条路径，并更新 `dist[i][j]`。
- 为了得到最终结果，需要让每个顶点都作为中转站遍历一次。


## 4. 数据结构

- 邻接矩阵 $(n*n) graph$，其中 $graph[i][j]$ 表示从顶点 `i` 到顶点 `j` 的最短路径。


## 5. 初始化

- 初始时，所有距离都设为无穷大。
- 如果 $i == j$，则 $graph[i][j] = 0$。
- 遍历图中的所有边 $i \rightarrow j$，令 $graph[i][j] = 边权$。


## 6. 求解过程

- 要找到从顶点 `i` 到顶点 `j` 的最短路径，可以通过让 `i` 经过其他顶点 `k` 再到达 `j` $(i \rightarrow k \rightarrow j)$ 的路径，与原路径 $(i \rightarrow j)$ 进行比较，从而更新 `dist[i][j]`。
- 为了得到准确的结果，需要每个顶点都作为中转站进行一次计算。


## 7. 示例

- 比如我们可以先只经过编号为 1 的顶点，比较 $i \rightarrow j$ 和 $i \rightarrow 1 \rightarrow j$，选择更短的一条路径，然后更新 $dist[i][j]$ 的最短路径。


In [2]:
n = 3
graph = [[0, 1, 4], [2, 0, 5], [3, 6, 0]]

for k in range(n):
	for i in range(n):
		for j in range(n):
			graph[i][j] = min(graph[i][j], graph[i][k] + graph[k][j])

graph

[[0, 1, 4], [2, 0, 5], [3, 4, 0]]

同理经过其他点，比如编号为2的点

In [3]:
for i in range(n):
	for j in range(n):
		graph[i][j] = min(graph[i][j], graph[i][2] + graph[2][j])

最后我们每个点都要经过：

In [4]:
for k in range(n):
	for i in range(n):
		for j in range(n):
			graph[i][j] = min(graph[i][j], graph[i][k] + graph[k][j])

因为每次 i --> j 的路径都更新为最优的而且都被保存下来，所以后面的结点要通过这个结点去往别的结点，就能保证是最优的。

> *原稿配图未随 notebook 保存，此处暂缺。*

定义邻接矩阵：


In [5]:
graph = [
    [0, 2, float('inf'), 6],
    [2, 0, 3, 2],
    [float('inf'), 3, 0, 2],
    [6, 2, 2, 0]
]

def dijkstra(graph, start):
    n = len(graph)
    dist = [float('inf')] * n
    dist[start] = 0
    visited = [False] * n
    while True:
        min_dist = float('inf')
        min_index = -1
        for i in range(n):
            if not visited[i] and dist[i] < min_dist:
                min_dist = dist[i]
                min_index = i
        if min_index == -1:
            break
        visited[min_index] = True
        for i in range(n):
            if not visited[i] and graph[min_index][i] > 0 and dist[min_index] + graph[min_index][i] < dist[i]:
                dist[i] = dist[min_index] + graph[min_index][i]
    return dist

print(dijkstra(graph, 0))

[0, 2, 5, 4]


如果要打印路径，我们就需要用一个二维数组parents记录每个结点的父结点。在找最短路的时候，更新父结点。最后递归寻找父结点，回溯打印。

In [6]:
import sys

sys.setrecursionlimit(100000000)

# 弗洛伊德算法
def floyd(graph, parents):
    n = len(graph)
    for k in range(n):
        for i in range(n):
            for j in range(n):
                if graph[i][k] + graph[k][j] < graph[i][j]:
                    graph[i][j] = graph[i][k] + graph[k][j]
                    parents[i][j] = parents[k][j]  # 更新父节点


def print_path(i, j):
    if i != j:
        print_path(i, parents[i][j])
    print(j, end='-->')

# 数据 [u, v, cost]
datas = [
    [0, 1, 2],
    [0, 2, 6],
    [0, 3, 4],
    [1, 2, 3],
    [2, 0, 7],
    [2, 3, 1],
    [3, 0, 5],
    [3, 2, 12],
]

n = 4

# 无穷大
inf = float('inf')

# 构图
graph = [[(0 if i == j else inf) for j in range(n)] for i in range(n)]
parents = [[i for _ in range(n)] for i in range(n)]  # 初始化父节点为自身

for u, v, c in datas:
    graph[u][v] = c  # 有向图

floyd(graph, parents)

print('Costs:')
for row in graph:
    for e in row:
        print('∞' if e == inf else e, end='\t')
    print()

print('\nPath:')
for i in range(n):
    for j in range(n):
        print('Path({}-->{}): '.format(i, j), end='')
        print_path(i, j)
        print(' cost:', graph[i][j])


Costs:
0	2	5	4	
9	0	3	4	
6	8	0	1	
5	7	10	0	

Path:
Path(0-->0): 0--> cost: 0
Path(0-->1): 0-->1--> cost: 2
Path(0-->2): 0-->1-->2--> cost: 5
Path(0-->3): 0-->3--> cost: 4
Path(1-->0): 1-->2-->3-->0--> cost: 9
Path(1-->1): 1--> cost: 0
Path(1-->2): 1-->2--> cost: 3
Path(1-->3): 1-->2-->3--> cost: 4
Path(2-->0): 2-->3-->0--> cost: 6
Path(2-->1): 2-->3-->0-->1--> cost: 8
Path(2-->2): 2--> cost: 0
Path(2-->3): 2-->3--> cost: 1
Path(3-->0): 3-->0--> cost: 5
Path(3-->1): 3-->0-->1--> cost: 7
Path(3-->2): 3-->0-->1-->2--> cost: 10
Path(3-->3): 3--> cost: 0


In [7]:
import numpy as np

INF = float('inf')

def floyd_warshall(graph):
    num_vertices = len(graph)
    
    dist = np.zeros((num_vertices, num_vertices))
    
    for i in range(num_vertices):
        for j in range(num_vertices):
            if i == j:
                dist[i][j] = 0
            elif graph[i][j] != 0:
                dist[i][j] = graph[i][j]
            else:
                dist[i][j] = INF

    for k in range(num_vertices):
        for i in range(num_vertices):
            for j in range(num_vertices):
                if dist[i][j] > dist[i][k] + dist[k][j]:
                    dist[i][j] = dist[i][k] + dist[k][j]
    

    return dist

# 示例图的邻接矩阵（0表示没有边，其他数字表示权重）
graph = [
    [0, 3, INF, INF, INF],
    [2, 0, INF, 1, INF],
    [INF, 7, 0, INF, 2],
    [6, INF, 12, 0, 5],
    [INF, INF, 4, INF, 0]
]

# 运行Floyd-Warshall算法
shortest_paths = floyd_warshall(graph)

# 打印结果
print("最短路径矩阵：")
print(shortest_paths)



最短路径矩阵：
[[ 0.  3. 13.  4.  9.]
 [ 2.  0. 10.  1.  6.]
 [ 9.  7.  0.  8.  2.]
 [ 6.  9.  9.  0.  5.]
 [13. 11.  4. 12.  0.]]


In [8]:
graph = {
    'A': [(7, 'A', 'B'), (5, 'A', 'D')],
    'B': [(7, 'B', 'A'), (8, 'B', 'C'), (9, 'B', 'D'), (7, 'B', 'E')],
    'C': [(8, 'C', 'B'), (5, 'C', 'E')],
    'D': [(5, 'D', 'A'), (9, 'D', 'B'), (15, 'D', 'E'), (6, 'D', 'F')],
    'E': [(7, 'E', 'B'), (5, 'E', 'C'), (15, 'E', 'D'), (8, 'E', 'F'), (9, 'E', 'G')],
    'F': [(6, 'F', 'D'), (8, 'F', 'E'), (11, 'F', 'G')],
    'G': [(9, 'G', 'E'), (11, 'G', 'F')]
}

def graph_to_adjacency_matrix(graph):
    vnum = len(graph)
    vertex_dict = {vertex: index for index, vertex in enumerate(graph.keys())}
    adjacent_matrix = [[float('inf') if row != col else 0 for col in range(vnum)] for row in range(vnum)]
    
    for vertex in graph:
        for edge in graph[vertex]:
            weight, u, v = edge
            # 无向图的对称性处理
            adjacent_matrix[vertex_dict[u]][vertex_dict[v]] = weight
            adjacent_matrix[vertex_dict[v]][vertex_dict[u]] = weight  # 添加对称边
            
    return adjacent_matrix

# 调用函数并打印结果
adjacency_matrix = graph_to_adjacency_matrix(graph)

# 打印邻接矩阵
for row in adjacency_matrix:
    print(row)


[0, 7, inf, 5, inf, inf, inf]
[7, 0, 8, 9, 7, inf, inf]
[inf, 8, 0, inf, 5, inf, inf]
[5, 9, inf, 0, 15, 6, inf]
[inf, 7, 5, 15, 0, 8, 9]
[inf, inf, inf, 6, 8, 0, 11]
[inf, inf, inf, inf, 9, 11, 0]


In [11]:
def graph2adjacent_matrix(graph):
    # 假设 graph 是一个字典，其中键是顶点，值是该顶点连接的所有顶点及其距离
    # 例如：graph = {'A': {'B': 1, 'C': 4}, 'B': {'A': 1, 'C': 2, 'D': 5}, ...}
    vertices = sorted(graph.keys())
    vnum = len(vertices)
    adjacent_matrix = [[float('inf')] * vnum for _ in range(vnum)]
    
    for i in range(vnum):
        for vertex, distance in graph[vertices[i]].items():
            j = vertices.index(vertex)
            adjacent_matrix[i][j] = distance
    
    # 添加自环
    for i in range(vnum):
        adjacent_matrix[i][i] = 0
    
    return adjacent_matrix


In [12]:
# def floyd(adjacent_matrix):
#     vnum = len(adjacent_matrix)
#     a = [[adjacent_matrix[row][col] for col in range(vnum)] for row in range(vnum)]
#     nvertex = [[-1 if adjacent_matrix[row][col]==float('inf') else col for col in range(vnum)] for row in range(vnum)]
#     # print(adjacent_matrix)
#     for k in range(vnum):
#         for i in range(vnum):
#             for j in range(vnum):
#                 if a[i][j]>a[i][k]+a[k][j]:
#                     a[i][j]=a[i][k]+a[k][j] 
#                     nvertex[i][j] = nvertex[i][k]
#     return nvertex, a

# adjacent_matrix = graph2adjacent_matrix(graph)
# nvertex, a = floyd(adjacent_matrix)


In [14]:
def graph2adjacent_matrix(graph):
    # 假设 graph 是一个字典，其中键是顶点，值是该顶点连接的所有顶点及其距离
    # 例如：graph = {'A': {'B': 1, 'C': 4}, 'B': {'A': 1, 'C': 2, 'D': 5}, ...}
    vertices = sorted(graph.keys())
    vnum = len(vertices)
    adjacent_matrix = [[float('inf')] * vnum for _ in range(vnum)]
    
    for i in range(vnum):
        for vertex, distance in graph[vertices[i]].items():
            j = vertices.index(vertex)
            adjacent_matrix[i][j] = distance
    
    # 添加自环
    for i in range(vnum):
        adjacent_matrix[i][i] = 0
    
    return adjacent_matrix

def floyd(adjacent_matrix):
    vnum = len(adjacent_matrix)
    a = [[adjacent_matrix[row][col] for col in range(vnum)] for row in range(vnum)]
    nvertex = [[-1 if adjacent_matrix[row][col] == float('inf') else col for col in range(vnum)] for row in range(vnum)]
    for k in range(vnum):
        for i in range(vnum):
            for j in range(vnum):
                if a[i][j] > a[i][k] + a[k][j]:
                    a[i][j] = a[i][k] + a[k][j]
                    nvertex[i][j] = nvertex[i][k]
    return nvertex, a

# 示例图定义
graph = {
    'A': {'B': 1, 'C': 4},
    'B': {'A': 1, 'C': 2, 'D': 5},
    'C': {'A': 4, 'B': 2, 'D': 1},
    'D': {'B': 5, 'C': 1}
}

# 将图转换为邻接矩阵
adjacent_matrix = graph2adjacent_matrix(graph)

# 使用 Floyd 算法计算最短路径和经过的顶点
nvertex, a = floyd(adjacent_matrix)

### 打印原邻接矩阵 ###
print("### 打印原邻接矩阵 ###")
for i in range(len(adjacent_matrix)):
    for j in range(len(adjacent_matrix[0])):
        print(adjacent_matrix[i][j], end="\t")
    print()  # 打印一行后换行

### 打印经过的顶点 ###
print("\n### 打印经过的顶点 ###")
for i in range(len(nvertex)):
    for j in range(len(nvertex[0])):
        print(nvertex[i][j], end="\t")
    print()  # 打印一行后换行

### 打印彼此之间的最短距离 ###
print("\n### 打印彼此之间的最短距离 ###")
for i in range(len(a)):
    for j in range(len(a[0])):
        print(a[i][j], end="\t")
    print()  # 打印一行后换行


### 打印原邻接矩阵 ###
0	1	4	inf	
1	0	2	5	
4	2	0	1	
inf	5	1	0	

### 打印经过的顶点 ###
0	1	1	1	
0	1	2	2	
1	1	2	3	
2	2	2	3	

### 打印彼此之间的最短距离 ###
0	1	3	4	
1	0	2	3	
3	2	0	1	
4	3	1	0	


In [17]:
# 创建一个示例图的邻接矩阵
# 这里假设图有4个顶点，并且边的权重如下
# 顶点1到顶点2的边权重为3，顶点1到顶点3的边权重为8，其他没有直接连接的顶点权重设为无穷大（float('inf')）
graph = {
    0: {1: 3, 2: 8},
    1: {0: 3, 3: 1},
    2: {0: 8, 3: 2},
    3: {1: 1, 2: 2}
}

def graph2adjacent_matrix(graph):
    vnum = len(graph)
    adjacent_matrix = [[float('inf')] * vnum for _ in range(vnum)]
    for i in range(vnum):
        for j in graph[i]:
            adjacent_matrix[i][j] = graph[i][j]
        adjacent_matrix[i][i] = 0  # 顶点到自身的距离为0
    return adjacent_matrix

adjacent_matrix = graph2adjacent_matrix(graph)
nvertex, a = floyd(adjacent_matrix)
### 打印原邻接矩阵 ###
for i in range(len(adjacent_matrix)):
    for j in range(len(adjacent_matrix[0])):
        print(adjacent_matrix[i][j], end="\t")
    print()  # 打印一行后换行
### 打印经过的顶点 ###
print()
for i in range(len(nvertex)):
    for j in range(len(nvertex[0])):
        print(nvertex[i][j], end="\t")
    print()  # 打印一行后换行
### 打印彼此之间的最短距离 ###
print()
for i in range(len(a)):
    for j in range(len(a[0])):
        print(a[i][j], end="\t")
    print()  # 打印一行后换行


0	3	8	inf	
3	0	inf	1	
8	inf	0	2	
inf	1	2	0	

0	1	1	1	
0	1	3	3	
3	3	2	3	
1	1	2	3	

0	3	6	4	
3	0	3	1	
6	3	0	2	
4	1	2	0	


In [15]:
import numpy as np

# 用于表示无穷大，避免整数溢出
max_value = 66666  

def floyd_warshall():
    # 初始化图的距离矩阵
    dist = np.array([
        [0, 2, 3, 6, max_value, max_value],
        [2, 0, max_value, max_value, 4, 6],
        [3, max_value, 0, 2, max_value, max_value],
        [6, max_value, 2, 0, 1, 3],
        [max_value, 4, max_value, 1, 0, max_value],
        [max_value, 6, max_value, 3, max_value, 0]
    ])
    
    # Floyd 算法核心计算
    num_vertices = dist.shape[0]  # 获取节点数
    for k in range(num_vertices):  # 对每个节点进行更新
        for i in range(num_vertices):  # 对每个起始节点
            for j in range(num_vertices):  # 对每个终止节点
                dist[i][j] = min(dist[i][j], dist[i][k] + dist[k][j])
    
    # 输出结果
    for i in range(num_vertices):
        print("节点{} 的最短路径: ".format(i + 1), end="")
        for j in range(num_vertices):
            print(dist[i][j], end=" ")
        print()  # 换行

# 调用函数
floyd_warshall()


节点1 的最短路径: 0 2 3 5 6 8 
节点2 的最短路径: 2 0 5 5 4 6 
节点3 的最短路径: 3 5 0 2 3 5 
节点4 的最短路径: 5 5 2 0 1 3 
节点5 的最短路径: 6 4 3 1 0 4 
节点6 的最短路径: 8 6 5 3 4 0 
